In [1]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle


In [3]:

df = pd.read_csv("Training.csv")
df.head()


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [4]:

X = df.drop(columns=['prognosis'])
Y = df['prognosis']
le = LabelEncoder()
Y = le.fit_transform(Y)


In [5]:

selector = SelectKBest(score_func=chi2, k=30)
X_new = selector.fit_transform(X, Y)


In [6]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_new)


In [7]:

x_train, x_test, y_train, y_test = train_test_split(X_scaled, Y, test_size=0.3, random_state=42)


In [8]:

params = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
grid = GridSearchCV(SVC(), params, cv=5)
grid.fit(x_train, y_train)
best_svc = grid.best_estimator_


In [9]:

rf = RandomForestClassifier()
knn = KNeighborsClassifier()
gb = GradientBoostingClassifier()

rf.fit(x_train, y_train)
knn.fit(x_train, y_train)
gb.fit(x_train, y_train)


GradientBoostingClassifier()

In [11]:

voting = VotingClassifier(estimators=[
    ('svc', best_svc),
    ('rf', rf),
    ('knn', knn)
], voting='hard')

voting.fit(x_train, y_train)


VotingClassifier(estimators=[('svc', SVC(C=0.1)),
                             ('rf', RandomForestClassifier()),
                             ('knn', KNeighborsClassifier())])

In [12]:

y_pred = voting.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.43089430894308944
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        32
           1       0.00      0.00      0.00        39
           2       0.00      0.00      0.00        41
           3       1.00      1.00      1.00        36
           4       0.00      0.00      0.00        35
           5       1.00      0.92      0.96        36
           6       1.00      0.93      0.96        44
           7       0.00      0.00      0.00        32
           8       1.00      1.00      1.00        35
           9       0.00      0.00      0.00        30
          10       1.00      1.00      1.00        31
          11       1.00      1.00      1.00        40
          12       1.00      1.00      1.00        33
          13       0.05      1.00      0.10        45
          14       0.00      0.00      0.00        35
          15       0.00      0.00      0.00        28
          16       0.00      0.00      0.00        

C:\Users\anand\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\anand\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\anand\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [13]:

with open("disease_model.pkl", "wb") as f:
    pickle.dump(voting, f)
